# synthpose-webots — Geração de dataset COCO-pose do NAO no Kaggle

Este notebook roda o pipeline de geração de dataset sintético (Blender 5.1.2 headless + Cycles GPU)
diretamente em um **Kaggle Notebook**.

## Antes de rodar (menu à direita do Kaggle)
1. **Settings → Accelerator → GPU** (T4 x2 ou P100). Necessário para o Cycles renderizar rápido.
2. **Settings → Internet → On**. Necessário para baixar o Blender e clonar o repositório.

## O que o notebook faz
1. Baixa e extrai o **Blender 5.1.2** (Linux x64).
2. Clona o repositório `synthpose-webots`.
3. Instala `pyyaml` no Python embutido do Blender (numpy já vem embutido; `cv2`/`pycocotools`
   só são usados na validação, que roda no Python do Kaggle).
4. Habilita a **GPU no Cycles** (OptiX → CUDA) via um wrapper.
5. Roda `dataset_generator_phobos.py` gravando em `/kaggle/working/output`.
6. Valida o JSON COCO e mostra uma amostra renderizada.
7. Compacta a saída para download.


## 1. Configuração
Ajuste `NUM_SAMPLES` conforme o tempo disponível. O Kaggle limita a sessão a ~9h de GPU;
comece pequeno (ex.: 20) para validar o pipeline antes de gerar um lote grande.

> **Repositório privado?** Se `git clone` falhar por autenticação, faça upload do repo como um
> *Kaggle Dataset* e aponte `REPO_DIR` para `/kaggle/input/<seu-dataset>/synthpose-webots`
> (pule a célula de clone).

In [ ]:
import os

# --- Blender ---
BLENDER_VERSION = "5.1.2"
BLENDER_SERIES  = "Blender5.1"
BLENDER_URL = f"https://download.blender.org/release/{BLENDER_SERIES}/blender-{BLENDER_VERSION}-linux-x64.tar.xz"

# --- Repositorio ---
REPO_URL    = "https://github.com/Rinaldots/synthpose-webots.git"
REPO_BRANCH = "main"
REPO_DIR    = "/kaggle/working/synthpose-webots"   # ajuste p/ /kaggle/input/... se usar Dataset

# --- Geracao ---
NUM_SAMPLES = 20                          # comece pequeno; aumente depois
START_INDEX = 0
OUT_DIR     = "/kaggle/working/output"    # persiste e fica disponivel p/ download
USE_GPU     = True

WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)
print("Config OK")

## 2. Baixar e extrair o Blender 5.1.2

In [ ]:
import os, shutil, subprocess, glob, tarfile, urllib.request

TARBALL = f"{WORK}/blender.tar.xz"
BLENDER_HOME = f"{WORK}/blender-{BLENDER_VERSION}-linux-x64"

if not os.path.isdir(BLENDER_HOME):
    # download.blender.org recusa User-Agent nao-navegador (HTTP 403);
    # por isso enviamos um User-Agent de navegador.
    if not os.path.exists(TARBALL) or os.path.getsize(TARBALL) < 1_000_000:
        print("Baixando Blender...", BLENDER_URL)
        req = urllib.request.Request(BLENDER_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(TARBALL, "wb") as f:
            shutil.copyfileobj(r, f)
    print("Extraindo...")
    with tarfile.open(TARBALL) as t:
        t.extractall(WORK)

BLENDER = os.path.join(BLENDER_HOME, "blender")
assert os.path.exists(BLENDER), f"blender nao encontrado em {BLENDER}"

# Bibliotecas de sistema que o Blender headless costuma exigir no Kaggle
subprocess.run("apt-get -qq update && apt-get -qq install -y libxrender1 libxi6 libxxf86vm1 "
               "libxfixes3 libxkbcommon0 libsm6 libgl1 libegl1 2>/dev/null || true",
               shell=True)

print(subprocess.run([BLENDER, "--version"], capture_output=True, text=True).stdout)

## 3. Clonar o repositório
Pule esta célula e ajuste `REPO_DIR` na célula 1 se o repo estiver como *Kaggle Dataset*.

In [ ]:
import os, subprocess, shutil

if REPO_DIR.startswith("/kaggle/working") and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],
                   check=True)

DG = os.path.join(REPO_DIR, "dataset_generator")
BLENDER_DIR = os.path.join(DG, "blender")
assert os.path.isfile(os.path.join(BLENDER_DIR, "nao_blender.blend")), \
    f"nao_blender.blend nao encontrado em {BLENDER_DIR}"
print("Repo pronto em:", REPO_DIR)

## 4. Instalar `pyyaml` no Python embutido do Blender
O gerador roda dentro do Blender, então a dependência precisa ir no Python **dele**
(não no do Kaggle). `numpy` já vem embutido.

In [ ]:
import glob, subprocess

BPY = glob.glob(os.path.join(BLENDER_HOME, "*", "python", "bin", "python3*"))
BPY = sorted(BPY)[0]
print("Python do Blender:", BPY)

subprocess.run([BPY, "-m", "ensurepip"], check=False)
subprocess.run([BPY, "-m", "pip", "install", "--upgrade", "pip"], check=False)
subprocess.run([BPY, "-m", "pip", "install", "pyyaml"], check=True)
print(subprocess.run([BPY, "-c", "import yaml, numpy; print('yaml', yaml.__version__, '| numpy', numpy.__version__)"],
                     capture_output=True, text=True).stdout)

## 5. Wrapper para habilitar a GPU no Cycles
O `dataset_generator_phobos.py` define o engine como CYCLES mas deixa o *device* em CPU.
Este wrapper é passado com um `--python` **antes** do gerador para ligar OptiX/CUDA.

In [ ]:
GPU_WRAPPER = os.path.join(BLENDER_DIR, "kaggle_enable_gpu.py")

with open(GPU_WRAPPER, "w") as f:
    f.write('''
import bpy

prefs = bpy.context.preferences.addons["cycles"].preferences
chosen = None
for dtype in ("OPTIX", "CUDA"):
    try:
        prefs.compute_device_type = dtype
    except TypeError:
        continue
    prefs.get_devices()
    if any(d.type == dtype for d in prefs.devices):
        chosen = dtype
        break

enabled = 0
if chosen:
    for d in prefs.devices:
        d.use = (d.type == chosen)
        enabled += int(d.use)
    for sc in bpy.data.scenes:
        sc.cycles.device = "GPU"

print(f"[KAGGLE-GPU] compute_device_type={chosen} devices_enabled={enabled}")
''')

print("Wrapper escrito em", GPU_WRAPPER)

## 6. Rodar o gerador
Grava imagens e anotações em `/kaggle/working/output`. Aumente `NUM_SAMPLES` na célula 1
para lotes maiores (o script suporta `--start` para retomar de onde parou).

In [ ]:
import subprocess

cmd = [BLENDER, "--background", "nao_blender.blend"]
if USE_GPU:
    cmd += ["--python", "kaggle_enable_gpu.py"]
cmd += ["--python", "dataset_generator_phobos.py",
        "--", "--num", str(NUM_SAMPLES), "--start", str(START_INDEX), "--out", OUT_DIR]

print(" ".join(cmd))
proc = subprocess.run(cmd, cwd=BLENDER_DIR, text=True)
print("\nreturn code:", proc.returncode)

## 7. Validar o JSON COCO e visualizar uma amostra

In [ ]:
import glob, subprocess, json

# Validacao (roda no Python do Kaggle; numpy/opencv/pycocotools ja disponiveis na imagem base)
val = os.path.join(DG, "scripts", "validate_dataset.py")
ann = glob.glob(os.path.join(OUT_DIR, "annotations", "person_keypoints_*.json"))
env = dict(os.environ, PYTHONPATH=os.path.join(DG, "src"))
for a in ann:
    print("===", os.path.basename(a), "===")
    subprocess.run(["python", val, a], env=env)

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

imgs = sorted(glob.glob(os.path.join(OUT_DIR, "images", "**", "*.png"), recursive=True))
print(f"{len(imgs)} imagens geradas")
if imgs:
    n = min(4, len(imgs))
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    axes = [axes] if n == 1 else axes
    for ax, p in zip(axes, imgs[:n]):
        ax.imshow(mpimg.imread(p)); ax.set_title(os.path.basename(p)); ax.axis("off")
    plt.tight_layout(); plt.show()

## 8. Compactar a saída para download
O `.zip` aparece na aba **Output** / **Data** do Kaggle para download.

In [ ]:
import shutil
zip_path = shutil.make_archive("/kaggle/working/nao_dataset", "zip", OUT_DIR)
print("Pronto:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")